# Exercise 1

In [19]:

import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [5]:
filename = "AIT403_Tutorial04_customer_dirty.csv"
filepath = os.path.join(os.getcwd(), filename)

df = pd.read_csv(filepath)
print(df)

  CustomerID    Age  Gender          City Income    JoinDate Subscribed  Spend
0       C001   29.0  Female  Kuala Lumpur   4500  2026-01-05        Yes  120.5
1       C002    NaN   male             KL   5200  05/01/2026          Y   98.0
2       C003  999.0  FEMALE        Penang   6100  2026/01/06         No  110.0
3       C004   41.0       M   Johor Bahru      -  2026-01-07          1  105.0
4       C005   35.0  female   Johor Bahru   7200  07-01-2026          0  850.0
5       C006   38.0    Male  Kuala lumpur   6800  2026-01-08        Yes  115.0
6       C006   38.0    Male  Kuala lumpur   6800  2026-01-08        Yes  115.0
7       C007   31.0    male        Penang   5900         NaN          N  102.0


# Exercise 2

In [ ]:
#Remove the exact duplicate rows
df_clean = df.drop_duplicates(df)

#Standardize gender into only male/female
df_clean["Gender"] = df_clean['Gender'].str.strip().str.title()
gender_map = {'M':'Male', 'F':'Female'}
df_clean["Gender"] = df_clean['Gender'].replace(gender_map)

#Standardize city
city_map = {'KL':'Kuala Lumpur', 'Kuala lumpur':'Kuala Lumpur'}
df_clean['City'] = df_clean['City'].replace(city_map)

#Convert income into numeric
df_clean['Income'] = pd.to_numeric(df_clean['Income'].replace('-', np.nan), errors='coerce')

#Convert JoinDate to datetime
df_clean['JoinDate'] = pd.to_datetime(df_clean['JoinDate'], format='mixed', dayfirst=True)

#Convert Subscribed into a consistent binary representations
sub_map = {'Yes': 1, 'Y': 1, '1': 1, '1.0': 1, 'No': 0, 'N': 0, '0': 0, '0.0': 0}
df_clean['Subscribed'] = df_clean['Subscribed'].astype(str).str.strip().map(sub_map).astype('Int64')

#Detect suspicious Age and Spend values.
df_clean.loc[df_clean['Age'] > 120, 'Age'] = np.nan

print("Info:")
df_clean.info()
print(f"\nNull Counts: {df_clean.isnull().sum()}")
print("\nDuplicates: ", df_clean.duplicated().sum())
print(df_clean.describe(include='all'))

Info:
<class 'pandas.core.frame.DataFrame'>
Index: 7 entries, 0 to 7
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   CustomerID  7 non-null      object        
 1   Age         5 non-null      float64       
 2   Gender      7 non-null      object        
 3   City        7 non-null      object        
 4   Income      6 non-null      float64       
 5   JoinDate    6 non-null      datetime64[ns]
 6   Subscribed  7 non-null      Int64         
 7   Spend       7 non-null      float64       
dtypes: Int64(1), datetime64[ns](1), float64(3), object(3)
memory usage: 511.0+ bytes

Null Counts: CustomerID    0
Age           2
Gender        0
City          0
Income        1
JoinDate      1
Subscribed    0
Spend         0
dtype: int64

Duplicates:  0
       CustomerID       Age Gender          City       Income  \
count           7   5.00000      7             7     6.000000   
unique          7       NaN     

C:\Users\user\AppData\Local\Temp\ipykernel_31152\2203361517.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean["Gender"] = df_clean['Gender'].str.strip().str.title()
C:\Users\user\AppData\Local\Temp\ipykernel_31152\2203361517.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean["Gender"] = df_clean['Gender'].replace(gender_map)
C:\Users\user\AppData\Local\Temp\ipykernel_31152\2203361517.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try us

# Exercise 3

In [18]:
cases = {
    "A (Age missing 2%, symmetric)": {
        "Nature": "MCAR (likely random)",
        "Decision": "Mean imputation",
        "Justification": "Distribution is symmetric, and 2% is a minimal loss. Mean preserves central tendency."
    },
    "B (Income missing 25%, self-employed bias)":{
        "Nature":"MNAR (Missing Not At Random)",
        "Decision":"Model-based imputation or indicate missingness",
        "Justification":" Missingness depends on employment status. A binary indicator column ('Income_Missing') should be created before imputing."
    },
    "C (Sensor time gaps)": {
        "Nature": "MCAR or MAR",
        "Decision": "Intepretation or Forward Fill",
        "Justification": "Time-series data carries temporal continuity. Adjacent seconds are highly correlated."
    },
    "D (Medical test for high-risk only)": {
        "Nature": "MNAR",
        "Decision": "Do not blindly impute",
        "Justification": "The absence of the test indicates 'low risk'. Imputing the mean of high-risk patience would artificially inflate the risk profile of healthy patients."
    },
    "E (Correlated numerical vars missing 10-15%)": {
        "Nature": "MAR",
        "Decision": "MICE (Multiple Imputation by Chained Equations)",
        "Justification": "Uses the correlation among multiple faetures to create accurate estimates rather than flat univariate imputation."
    }
}

for case, details in cases.items():
    print(f"\nCase {case}:")
    print(f"  - Missingness: {details['Nature']}")
    print(f"  - Action: {details['Decision']}")
    print(f"  - Why: {details['Justification']}")   
    
print("\n16. Poor Imputation Bias:")
print("Imputing median for Case B (Income) without adding a missing-indicator would erase the critical signalt that 'self_employment' people behave differently, biasing the model against finding self-employmed trends.")


Case A (Age missing 2%, symmetric):
  - Missingness: MCAR (likely random)
  - Action: Mean imputation
  - Why: Distribution is symmetric, and 2% is a minimal loss. Mean preserves central tendency.

Case B (Income missing 25%, self-employed bias):
  - Missingness: MNAR (Missing Not At Random)
  - Action: Model-based imputation or indicate missingness
  - Why:  Missingness depends on employment status. A binary indicator column ('Income_Missing') should be created before imputing.

Case C (Sensor time gaps):
  - Missingness: MCAR or MAR
  - Action: Intepretation or Forward Fill
  - Why: Time-series data carries temporal continuity. Adjacent seconds are highly correlated.

Case D (Medical test for high-risk only):
  - Missingness: MNAR
  - Action: Do not blindly impute
  - Why: The absence of the test indicates 'low risk'. Imputing the mean of high-risk patience would artificially inflate the risk profile of healthy patients.

Case E (Correlated numerical vars missing 10-15%):
  - Missin

# Exercise 4

In [ ]:
df_features = pd.DataFrame({
    'Age': [29, 35, 41, 31, 38],
    'Income': [4500, 5200, 6100, 7200, 6000],
    'Spend': [120.5, 98.0, 110.0, 850.0, 115.0]
})

scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(df_features)
print(f"\nScaled Features (Age, Income, Spend):\n", scaled_features[:2])

df_features['Spend_to_Income_Ratio'] = df_features['Spend'] / df_features['Income']
print(" - Feature 1: Spend_to_Income_Ratio (Helps identify high-value/reckless spenders irrespective of absolute income).")

df_features['Days_Since_Join'] = 290
print(" - Feature 2: Days_Since_Join (Duration of relationship usually strongly correlates with subscription loyalty).")

